### 0. 事前準備

In [1]:
# 基本の標準ライブラリ
import os
import time
import requests
import json

In [2]:
import torch

In [3]:
# Nvidia GPUが利用できるか確認。
torch.cuda.is_available()

True

In [4]:
# データ分析系（現状、pandasのみ使用）
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

In [5]:
# PythonからPostgresSQLを操作するためのライブラリ
import psycopg2

In [6]:
# 文章の埋め込みのライブラリ
from sentence_transformers import SentenceTransformer

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
# 文章の埋め込みのモデルをロード
model_path = "/sentence_embedding_models/static-embedding-japanese/"
embedding_model = SentenceTransformer(model_path, device="cuda")

In [8]:
# カレントディレクトリ確認（実行は任意）
os.getcwd()

'/jupyterlab/workspace'

### 1. ベクトルデータベースへの文章登録

#### 1-1. データベース接続と拡張機能有効化

In [9]:
# Postgres接続
for i in range(10):
    try:
        conn = psycopg2.connect(
            dbname="ragdb",
            user="postgres",
            password="postgres",
            host="postgres_pgvector",
            port="5432"
        )
        print("DB接続完了！")
        break
    except Exception as e:
        print("DB接続待機中...", e)
        time.sleep(3)

cur = conn.cursor()

DB接続完了！


In [10]:
# 初回のみ実行。
# ベクトルデータベース用の拡張機能を有効にする。
cur.execute("CREATE EXTENSION IF NOT EXISTS vector")

#### 1-2. 文章登録

In [11]:
# テーブル作成
cur.execute("CREATE TABLE IF NOT EXISTS vector_table (id SERIAL PRIMARY KEY, content TEXT, embedding vector(1024))")

In [12]:
# サンプル文書をDBへ登録
texts = [
    "RAGは検索と生成を組み合わせたAI技術です。",
    "llama.cppはLLaMAモデルをC++で動かす実装です。",
    "gpt-ossは、2025年8月に公表されました。"
]
for t in texts:
    emb = embedding_model.encode(t).tolist()
    cur.execute("INSERT INTO vector_table (content, embedding) VALUES (%s, %s::vector)", (t, emb))

conn.commit()
print("✅ サンプル文書をDBに登録しました。")

✅ サンプル文書をDBに登録しました。


In [13]:
# 現在のテーブルの中身を取得
cur.execute("SELECT * FROM vector_table")
rows = cur.fetchall()

In [14]:
# pandasのDataFrameとして、SELECT文の結果を読み込み、表示
df = pd.DataFrame(rows, columns=[column.name for column in cur.description])
print(df.shape)
df.head()

(3, 3)


,id,content,embedding
0,1,RAGは検索と生成を組み合わせたAI技術です。,"[0.13871585,2.1914806,0.23719923,-0.63374776,1..."
1,2,llama.cppはLLaMAモデルをC++で動かす実装です。,"[-0.310665,1.6995026,-2.9435565,-0.24335124,0...."
2,3,gpt-ossは、2025年8月に公表されました。,"[-0.21954979,-0.17037459,0.28188685,-0.5598576..."


In [68]:
# （必要に応じて実行。）データベースのレコードを削除する。
cur.execute("DELETE FROM vector_table WHERE content ='gpt-ossは、2025年8月に公表されました。'")

In [15]:
# （必要に応じて実行。）テーブルを削除。
cur.execute("DROP TABLE vector_table")

### 2. チャット実装

In [15]:
# 質問例
question = "gpt-ossが公表された時期を教えてください。"
print("ユーザーからの質問 (question) ")
print(question)

ユーザーからの質問 (question) 
gpt-ossが公表された時期を教えてください。


In [16]:
# 質問文章の埋め込み
question_emb = embedding_model.encode(question).tolist()
print("質問文章の埋め込み次元数：", len(question_emb))

質問文章の埋め込み次元数： 1024


In [17]:
# ベクトルデータベースから検索抽出する文章数
context_num = 2

In [18]:
# ベクトルデータベースの中から類似度が大きいデータを抽出
cur.execute("""
    SELECT content, embedding
    FROM vector_table
    ORDER BY embedding <=> %s::vector
    LIMIT %s;
""", ((question_emb,), context_num))
docs = cur.fetchall()
contexts = "\n".join([d[0] for d in docs])
print("ベクトルデータベースから抽出したcontext")
context_list = contexts.split("\n")
for i, context in enumerate(context_list):
    print("類似度", i + 1, "位", ": ", context)

ベクトルデータベースから抽出したcontext
類似度 1 位 :  gpt-ossは、2025年8月に公表されました。
類似度 2 位 :  llama.cppはLLaMAモデルをC++で動かす実装です。


In [19]:
# LLMに渡すプロンプトを作成
prompt = f"""あなたは与えられた情報だけを使って回答するアシスタントです。

◆ 重要ルール（厳守）
1. 以下の「情報」に記載されていない内容については絶対に回答しない。
2. 「参照情報」を明確にして回答する。
3. 情報に存在しない内容を推測・補完・創作してはならない。
4. 情報だけでは答えられない場合は次のように答える：
   「知りません。」

◆ 情報
{contexts}

"""

print("ベクトルデータベースから検索抽出した文章")
for i, context in enumerate(context_list):
    print(i+1, ": ", context)
print()

print("=== LLMへの入力プロンプト ===")
print(prompt)
print()

print("=== ユーザーからの質問文 ===")
print(question)
print()

ベクトルデータベースから検索抽出した文章
1 :  gpt-ossは、2025年8月に公表されました。
2 :  llama.cppはLLaMAモデルをC++で動かす実装です。

=== LLMへの入力プロンプト ===
あなたは与えられた情報だけを使って回答するアシスタントです。

◆ 重要ルール（厳守）
1. 以下の「情報」に記載されていない内容については絶対に回答しない。
2. 「参照情報」を明確にして回答する。
3. 情報に存在しない内容を推測・補完・創作してはならない。
4. 情報だけでは答えられない場合は次のように答える：
   「知りません。」

◆ 情報
gpt-ossは、2025年8月に公表されました。
llama.cppはLLaMAモデルをC++で動かす実装です。



=== ユーザーからの質問文 ===
gpt-ossが公表された時期を教えてください。



In [20]:
# 時間を計測
start_time = time.time()

# llama.cppサーバーのエンドポイント
LLAMA_API_URL = "http://llama-cpp:8080/v1/chat/completions" # UI画面からプロンプトを与えるときのエンドポイント

# llama.cppへリクエスト
payload = {
    "model": "Llama-3-ELYZA-JP-8B",
    "messages": [
        {"role": "system", "content": prompt},
        {"role": "user", "content": question},
    ],
    "temperature": 0.2, # 創造性の調整
    "max_tokens": 512
    }
print("\n=== llama.cpp に問い合わせ中... ===")
response = requests.post(LLAMA_API_URL, json=payload)

# JSONレスポンスをパースして出力
if response.ok:
    data = response.json()
    # llama.cpp は incremental streaming出力を返す場合があるので、'content'または'text'を確認
    try: # エンドポイントが/v1/chat/completionsの場合
        output = data["choices"][0]["message"]["content"]
    except Exception:
        output = "（出力を取得できませんでした）"
    print("\n=== ここからがLLMの回答 ===")
    print(output)
else:
    print(f"エラー: {response.status_code} - {response.text}")

print("ローカルLLM応答時間: ", time.time() - start_time)


=== llama.cpp に問い合わせ中... ===

=== ここからがLLMの回答 ===
gpt-ossは、2025年8月に公表されました。
ローカルLLM応答時間:  2.676375389099121
